In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

# Configure sua chave de API do Google (recomendado usar variável de ambiente)
api_key = os.environ.get('GOOGLE_API_KEY')
if not api_key:
    raise RuntimeError('Defina a variável de ambiente GOOGLE_API_KEY antes de executar este notebook.')

genai.configure(api_key=api_key)

# Caminho para a pasta de imagens
img_dir = Path('img')
images = list(img_dir.glob('*'))

print(f'Encontradas {len(images)} imagens em {img_dir.resolve()}')
for i, p in enumerate(images, start=1):
    print(f'{i}. {p.name} ({p.stat().st_size} bytes)')

Encontradas 4 imagens em C:\Programacao\nlw-python-image-recognition\mediapipe-gesture-recognition\img
1. bird.jpg (1788588 bytes)
2. cats_dog.png (709177 bytes)
3. kitchen.png (509182 bytes)
4. pizza.png (609616 bytes)


In [ ]:
import json

# Escolha uma imagem para enviar
if not images:
    raise RuntimeError('Nenhuma imagem encontrada em img/')

# Chamada à API Gemini
model = genai.GenerativeModel('gemini-2.5-flash')  # Modelo compatível com imagens

results = {}
for image_path in images:
    print(f'Enviando imagem: {image_path.name}')
    image_data = image_path.read_bytes()
    mime_type = 'image/jpeg' if image_path.suffix.lower() in ['.jpg', '.jpeg'] else 'image/png'
    
    response = model.generate_content([
        {'mime_type': mime_type, 'data': image_data},
        'Classifique a imagem em uma das seguintes categorias: "Cachorro", "Gato", "Pássaro", "Outro". Responda apenas com a categoria.'
    ])
    
    results[image_path.name] = response.text.strip() 

print('--- Resposta ---')
print(json.dumps(results, indent=2, ensure_ascii=False))

Enviando imagem: bird.jpg
Enviando imagem: cats_dog.png
Enviando imagem: kitchen.png
Enviando imagem: pizza.png
--- Resposta ---
{
  "bird.jpg": "Pássaro",
  "cats_dog.png": "Cachorro",
  "kitchen.png": "Outro",
  "pizza.png": "Outro"
}
